In [ ]:
SELECT DISTINCT
    CONCAT(
        tt.type, ' ',
        at.name
    ) AS cprod_name,
    'MPB001' AS cprod_src_sys_inst_id,
    CONCAT(
        CAST(a.treatment_type_id AS STRING),
        '_',
        CAST(a.appointment_type_id AS STRING)
    ) AS cprod_src_id
FROM silver_drj_appointments a
LEFT JOIN silver_drj_therapy_types tt
    ON a.treatment_type_id = tt.id
LEFT JOIN silver_drj_appointment_types at
    ON a.appointment_type_id = at.id
WHERE a.treatment_type_id IS NOT NULL
  AND a.appointment_type_id IS NOT NULL
  AND tt.id IS NOT NULL
  AND at.id IS NOT NULL
LIMIT 100

hhh

In [ ]:
WITH mpb_care_product AS (
    SELECT DISTINCT
        CONCAT(
            tt.type, ' ',
            at.name
        ) AS cprod_name,
        'MPB001' AS cprod_src_sys_inst_id,
        CONCAT(
            CAST(a.treatment_type_id AS STRING),
            '_',
            CAST(a.appointment_type_id AS STRING)
        ) AS cprod_src_id
    FROM silver_drj_appointments a
    LEFT JOIN silver_drj_therapy_types tt
        ON a.treatment_type_id = tt.id
    LEFT JOIN silver_drj_appointment_types at
        ON a.appointment_type_id = at.id
    WHERE a.treatment_type_id IS NOT NULL
      AND a.appointment_type_id IS NOT NULL
      AND tt.id IS NOT NULL
      AND at.id IS NOT NULL
)

SELECT
    m.cprod_name,
    m.cprod_src_sys_inst_id,
    m.cprod_src_id
FROM mpb_care_product m
LEFT JOIN silver_rdm_care_product r
    ON m.cprod_src_sys_inst_id = r.cprod_src_sys_inst_id
   AND m.cprod_src_id = r.cprod_src_id
WHERE r.cprod_src_id IS NULL

In [ ]:
CREATE OR REPLACE TABLE silver_rdm_care_product_add AS

WITH wip_care_product AS (
    SELECT DISTINCT
        -- WIP: cprod_name = Service Type + Service + Activity Type
        CONCAT(ast.description, ' ', srv.description, ' ', atp.description) AS cprod_name,
        -- Temporary hardcoded source system instance for WIP
        'WIP001' AS cprod_src_sys_inst_id,
        -- WIP: cprod_src_id = Service Type ID + Service ID + Activity Type ID
        CONCAT(CAST(ah.service_type_id AS STRING), '_', CAST(asv.service_id AS STRING), '_', CAST(ae.activity_type_id AS STRING)) AS cprod_src_id
    FROM silver_wip_activityentry ae
    LEFT JOIN silver_wip_activityheader ah ON ae.activity_header_id = ah.id
    LEFT JOIN silver_wip_activityservice asv ON ae.activity_service_id = asv.id
    LEFT JOIN silver_wip_servicetype ast ON ah.service_type_id = ast.id
    LEFT JOIN silver_wip_service srv ON asv.service_id = srv.id
    LEFT JOIN silver_wip_activitytype atp ON ae.activity_type_id = atp.id
    WHERE asv.is_primary = true -- Use only primary service mapping for WIP
      AND ah.id IS NOT NULL
      AND ah.service_type_id IS NOT NULL
      AND asv.service_id IS NOT NULL
      AND ae.activity_type_id IS NOT NULL
),

mpb_care_product AS (
    SELECT DISTINCT
        -- MPB: cprod_name = Treatment Type + Pathway
        CONCAT(tt.type, ' ', apt.name) AS cprod_name,
        -- Source system instance for MPB
        'MPB001' AS cprod_src_sys_inst_id,
        -- MPB: cprod_src_id = Treatment Type ID + Pathway ID
        CONCAT(CAST(a.treatment_type_id AS STRING), '_', CAST(a.appointment_type_id AS STRING)) AS cprod_src_id
    FROM silver_drj_appointments a
    LEFT JOIN silver_drj_therapy_types tt ON a.treatment_type_id = tt.id
    LEFT JOIN silver_drj_appointment_types apt ON a.appointment_type_id = apt.id
    WHERE a.treatment_type_id IS NOT NULL
      AND a.appointment_type_id IS NOT NULL
      AND tt.id IS NOT NULL
      AND apt.id IS NOT NULL
),

combined_care_product AS (
    -- Combine WIP and MPB derived care product records
    SELECT cprod_name, cprod_src_sys_inst_id, cprod_src_id FROM wip_care_product
    UNION
    SELECT cprod_name, cprod_src_sys_inst_id, cprod_src_id FROM mpb_care_product
)

-- Keep only new records not already present in silver_rdm_care_product
SELECT c.cprod_name, c.cprod_src_sys_inst_id, c.cprod_src_id
FROM combined_care_product c
LEFT JOIN silver_rdm_care_product r ON c.cprod_src_sys_inst_id = r.cprod_src_sys_inst_id AND c.cprod_src_id = r.cprod_src_id
WHERE r.cprod_src_id IS NULL

In [ ]:
finalized Care Product derivation for both WIP and MPB based on Monday definitions. For WIP, cprod_name is derived using Service Type + Service Activity + Activity Type and cprod_src_id using Service Type ID + Service Activity ID + Activity Type ID, with WIP001 as the temporary source system instance id. For MPB, cprod_name is derived using Treatment Type + Pathway and cprod_src_id using Treatment Type ID + Pathway ID, with MPB001 as the source system instance id. Prepared a combined query to create silver_rdm_care_product_add with only the required 3 columns and exclude records already present in silver_rdm_care_product.

In [ ]:
Hi, I created the silver_rdm_care_product_add semantic model using Direct Lake on SQL, same as the existing pattern.
I checked the older silver_rdm_service_add model settings and noticed that “Keep your Direct Lake data up to date” is turned Off there, while in the new model it is currently On by default.

Should I keep it On, or switch it Off to match the existing working setup?